# 3) XGboot prep
- This code generates the baseline model and splits the data into train/validation/test sets.
- It produces spearman clustering to extract different feature combinations.


# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import math, pickle
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

# 2. Reading and Cleaning data:

## 2-1. Opening data:

In [ ]:
df = pd.read_parquet('3_output/2_final_herd_detrend_df.gzip')
df.columns

In [ ]:
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(4,3), tight_layout=True)
df.groupby(['adj_year'])['herd_milk_resid'].mean().plot(ax=ax)
ax.legend('')
ax.set_title('Cow-level yearly milk residuals')

## 2-2) Formatting:

In [ ]:
## control variables:
# Compute sin/cos transformations for month
df['month'] = df['date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

## creating lac_num:
df['lac_bin'] = df['lac_num'].apply(lambda x: 1 if x == 1 else (2 if x == 2 else 3))
df['lac_dim'] = df['dim']
df.loc[df['lac_bin'] == 2, 'lac_dim'] = df.loc[df['lac_bin'] ==2]['dim'] + 305
df.loc[df['lac_bin'] == 3, 'lac_dim'] = df.loc[df['lac_bin'] ==3]['dim'] + 610

target = 'herd_milk_resid'

In [ ]:
df['vpdmean'] = (df['vpdmin'] + df['vpdmax'] ) /2
df['rh'] = (df['rh_am'] + df['rh_pm'])/2
df['ag_rh'] = (df['ag_rh_am'] + df['ag_rh_pm']) /2
df['tmean'] = (df['tmax'] + df['tmin']) /2
df['ag_tmean'] = (df['ag_tmax'] + df['ag_tmin']) /2
df['tmax_ssrd'] = ((df['tmax']+273.15) * df['ag_ssrd_wm-2']) /1000
df['tmean_ssrd']= ((df['tmean']+273.15) *df['ag_ssrd_wm-2'])/1000

In [ ]:
## saving:
df.to_parquet('3_output/3_final_herd_detrend_df_full.gzip',compression='gzip')

# 3. Baseline model (only biological cycles)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
df['ln_dim'] = np.log(df['dim'])
df['lac_bin'] = df['lac_bin'].astype(str)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False)  # Drop first to avoid dummy variable trap
encoded_categorical = encoder.fit_transform(df[['lac_bin']])

# Convert encoded variables into a DataFrame
encoded_df = pd.DataFrame(encoded_categorical, columns=encoder.get_feature_names_out(['lac_bin']))

# Creating interaction terms:
for col in encoded_df.columns:
    encoded_df[f'ln_dim:{col}'] = df['ln_dim'] * encoded_df[col]
    encoded_df[f'dim:{col}'] = df['dim'] * encoded_df[col]

In [ ]:
df = pd.concat([df, encoded_df], axis=1)
del encoded_df
gc.collect()

### a. training:

In [ ]:
expanding_splits = [(2006,2009),
                   (2009,2012),
                   (2012,2015),
                   (2015,2018),
                   (2018,2021)]

In [ ]:
train_output = pd.DataFrame()
val_output = pd.DataFrame()

In [ ]:
x_col = ['lac_bin_2','lac_bin_3',
         'ln_dim:lac_bin_2', 'dim:lac_bin_2',
         'ln_dim:lac_bin_3', 'dim:lac_bin_3']

rows = []

for idx, (train_end, val_end) in enumerate(expanding_splits, start=1):
    print(idx ,'--------------------------------', 'train_end', train_end, 'val_end', val_end)
        
    sub_target = f'herd_milk_resid_{idx}'
    
    ## masking:
    train_mask = df["adj_year"] <= train_end
    val_mask = (df["adj_year"] > train_end) & (df["adj_year"] <= val_end)

    ## creating subset:
    x_train = df.loc[train_mask][x_col].copy()
    y_train = df.loc[train_mask][sub_target]
    
    ## training model:
    base_model = LinearRegression(n_jobs=-1).fit(x_train, y_train)
    # print("Coefficients:", dict(zip(x_col, base_model.coef_)))
    
    ## predicted values:
    ptrain = base_model.predict(x_train)
    train_metrics = eval_util_module.evaluate(y_train, ptrain)
    rows.append({"fold": idx, "category": "train", "feature": "baseline", **train_metrics})
    
    ## validation:
    x_val = df.loc[val_mask, x_col]
    y_val = df.loc[val_mask, sub_target]

    pval = base_model.predict(x_val)
    val_metrics = eval_util_module.evaluate(y_val, pval)
    rows.append({"fold": idx, "category": "val", "feature": "baseline", **val_metrics})
    
    del base_model
    

## saving outputs:   
output = pd.DataFrame(rows)
train_output = output.loc[output["category"] == "train"].reset_index(drop=True)
val_output = output.loc[output["category"] == "val"].reset_index(drop=True)

### b. test:

In [ ]:
## for test:
for idx, (train_end, val_end) in enumerate(([2021,2024],[2024,2027])):
    print(idx, train_end)
    
    if train_end == 2021:
        print(idx ,'--------------------------------', 'train_end', train_end, 'val_end', val_end)
        
        sub_target = f'herd_milk_resid_train'

        ## masking:
        train_mask = df["adj_year"] <= train_end
        val_mask = (df["adj_year"] > train_end) & (df["adj_year"] <= val_end)

        ## creating subset:
        x_train = df.loc[train_mask][x_col].copy()
        y_train = df.loc[train_mask][sub_target]
    
        base_model = LinearRegression(n_jobs=-1).fit(x_train, y_train)
        print("Coefficients:", dict(zip(x_train.columns, base_model.coef_)))

        ptrain = base_model.predict(x_train)
        train_error = pd.DataFrame(eval_util_module.evaluate(y_train, ptrain), index=[idx])
        
        ## validation:
        x_val = df.loc[val_mask, x_col]
        y_val = df.loc[val_mask, sub_target]

        pval = base_model.predict(x_val)
        val_error = pd.DataFrame(eval_util_module.evaluate(y_val, pval), index=[idx])
        val_error["feature"] = "baseline"
        val_output = pd.concat([val_output, val_error])

        train_error["feature"] = "baseline"
        train_output = pd.concat([train_output, train_error])

    elif train_end == 2024:
        sub_target = "herd_milk_resid"

        X = df[x_col].copy()
        y = df[sub_target]

        mask = X.notna().all(axis=1) & y.notna()
        X = X.loc[mask]
        y = y.loc[mask]

        base_model = LinearRegression(n_jobs=-1).fit(X, y)
        ptrain = base_model.predict(X)

        # if you want baseline_fit aligned to original df index, write to those rows
        df.loc[mask, "baseline_fit"] = ptrain

        train_error = pd.DataFrame(eval_util_module.evaluate(y, ptrain), index=[idx])
        train_error["feature"] = "baseline"
        train_output = pd.concat([train_output, train_error])

        del base_model, ptrain, X, y
        gc.collect()

In [ ]:
base_output = pd.concat([train_output, val_output], axis=0, ignore_index=True)
base_output.loc[5,'category'] ='full_train'
base_output.loc[6,'category'] = 'full'
base_output.loc[12,'category'] = 'test'

In [ ]:
## saving baseline output:
base_output.to_csv('3_output/3_1_baseline_error_score.csv')

In [ ]:
df = df.drop(columns=['ln_dim', 'lac_bin_2', 'lac_bin_3',
       'ln_dim:lac_bin_2', 'dim:lac_bin_2', 'ln_dim:lac_bin_3',
       'dim:lac_bin_3'])

# 4. Train/Test Split
- This split should be based on adj_year, as the full lactation cycle should be included. Otherwise, this will create data leakage.

## 4-1. Checking train/test split
- 80% for training data and 20% for test data

In [ ]:
train_end = 2021
## train 80% of data and test 20% of data:
train = df[df['adj_year'] <= train_end].copy()
test = df[df['adj_year'] > train_end].copy()

In [ ]:
## saving them:
train = train.sort_values(by=['adj_year','state_abv','GEOID','geoid_herd','id','date'], ignore_index=True)
train.to_parquet('3_output/3_2_temp_train_avg_fit.gzip',compression='gzip')

test = test.sort_values(by=['adj_year','state_abv','GEOID','geoid_herd','id','date'], ignore_index=True)
test.to_parquet('3_output/3_2_temp_test_avg_fit.gzip',compression='gzip')

## 4-2. Cross-validation splits

In [ ]:
expanding_splits = [(2006,2009),
                   (2009,2012),
                   (2012,2015),
                   (2015,2018),
                   (2018,2021)]

In [ ]:
## check stats:
# fig, ax = plt.subplots(figsize=(12,5), nrows=2, ncols=3)
# axs=ax.flatten()

# for idx, (train_end, val_end) in enumerate(expanding_splits):
#     idx +=1
#     sub_target = f'herd_milk_resid_{idx}'    
#     print('length :', len(train.loc[(train['adj_year'] > train_end) & (train['adj_year'] <= val_end)][sub_target]))
#     sns.kdeplot(train.loc[(train['adj_year'] <= train_end)].groupby(['GEOID','adj_year','month'])[sub_target].mean(), ax=axs[idx])
#     sns.kdeplot(train.loc[(train['adj_year'] > train_end) & (train['adj_year'] <= val_end)].groupby(['GEOID','adj_year','month'])[sub_target].mean(), ax=axs[idx], linestyle='--')

# 5. Feature combinations

## 5-1. Spearman Hierachial Clustering

In [ ]:
param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim'],
         
         'clim_only':['tmax', 'tmin', 'tdmean', 'vpdmax', 'vpdmin', 'vpdmean','ppt', 'tmean', 'rh',
       'rh_pm', 'rh_am', 'wetbT', 'thi_max', 'thi_min', 'thi_avg',
       'adjthi', 'ag_tmax', 'ag_tmin', 'ag_tmean', 'ag_tdmean', 'ag_rh_am',
       'ag_rh_pm', 'ag_rh','ag_ppt', 'ag_ssrd_wm-2', 'ag_wind_2m',
       'ag_wetbT',  'ag_thi_max', 'ag_thi_min', 'ag_adjthi','tmax_ssrd'],
         
         'clim_only1': ['tmax', 'tmin', 'tdmean', 'vpdmax', 'vpdmin', 'ppt',
       'rh_pm', 'rh_am', 'wetbT', 'thi_max', 'thi_min',
       'adjthi', 'ag_tmax', 'ag_tmin', 'ag_tdmean', 'ag_rh_am',
       'ag_rh_pm', 'ag_ppt', 'ag_ssrd_wm-2', 'ag_wind_2m',
       'ag_wetbT', 'ag_thi_max', 'ag_thi_min', 'ag_adjthi'],
         
        } 


In [ ]:
from scipy.stats import spearmanr
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
feature = train.loc[:,train.columns.isin(param['clim_only'])].columns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
corr = spearmanr(train[feature]).correlation

# Ensure the correlation matrix is symmetric
corr = (corr + corr.T) / 2
np.fill_diagonal(corr, 1)

# We convert the correlation matrix to a distance matrix before performing
# hierarchical clustering using Ward's linkage.
distance_matrix = 1 - np.abs(corr)
dist_linkage = hierarchy.ward(squareform(distance_matrix))

dendro = hierarchy.dendrogram(
    dist_linkage, labels=feature, 
    ax=ax1, leaf_rotation=90
)
dendro_idx = np.arange(0, len(dendro["ivl"]))
ax1.tick_params(labelsize=13)


ax2.imshow(corr[dendro["leaves"], :][:, dendro["leaves"]], vmin=-1, vmax=1, cmap='RdBu_r')
ax2.set_xticks(dendro_idx)
ax2.set_yticks(dendro_idx)
ax2.set_xticklabels(dendro["ivl"], rotation="vertical")
ax2.set_yticklabels(dendro["ivl"])
ax2.tick_params(labelsize=12)
fig.tight_layout()
plt.savefig('3_output/fig/SI_spearman_clustering.png',dpi=300)
plt.show()

In [ ]:
from collections import defaultdict
cluster_ids = hierarchy.fcluster(dist_linkage, 0.6, criterion="distance")
cluster_id_to_feature_ids = defaultdict(list)
for idx, cluster_id in enumerate(cluster_ids):
    cluster_id_to_feature_ids[cluster_id].append(idx)
spear_features = [v[0] for v in cluster_id_to_feature_ids.values()]
train[feature].columns[spear_features]

In [ ]:
print('group1 :', train.loc[:,train.columns.isin(param['clim_only'])].columns[cluster_id_to_feature_ids[1]],
      'group2 :', train.loc[:,train.columns.isin(param['clim_only'])].columns[cluster_id_to_feature_ids[4]],
      'group3 :', train.loc[:,train.columns.isin(param['clim_only'])].columns[cluster_id_to_feature_ids[3]],
      'group4 :', train.loc[:,train.columns.isin(param['clim_only'])].columns[cluster_id_to_feature_ids[2]],
     'group5 :', train.loc[:,train.columns.isin(param['clim_only'])].columns[cluster_id_to_feature_ids[5]])
